In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1. 读取 54 个完整数据
df = pd.read_excel('EG-3.xlsx').reset_index(drop=True)

# 2. 定义特征 X 和目标 y 提取函数
def get_X_y(data_frame, target_col='Erxn'):
    # 请确保 drop 掉所有非特征列/目标列（如 Erxn, Eads, Name 等）
    X = data_frame.drop(columns=['Erxn', 'Eads', 'Name'], errors='ignore')
    y = data_frame[target_col]
    return X, y

cv_rmse_scores = []

# 3. 蒙特卡洛 100 次模拟
for i in range(100):
    # 第一步：从 54 个数据中随机抽取 18 个子集
    df_18_selected, _ = train_test_split(
        df, 
        train_size=36, 
        random_state=i, 
        shuffle=True
    )
    
    # 第二步：将这 18 个数据进一步拆分为训练集（如 14 个）和测试集（如 4 个）
    df_train_14, df_test_4 = train_test_split(
        df_18_selected, 
        test_size=8,        # 4 个作为测试集，剩余 14 个作为训练集
        random_state=i, 
        shuffle=True
    )
    
    # 提取特征与目标值
    X_train, y_train = get_X_y(df_train_14, target_col='Erxn') # 评估吸附能改为 'Eads'
    X_test, y_test = get_X_y(df_test_4, target_col='Erxn')
    
    # 训练 GBR 模型
    model = GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=i
    )
    model.fit(X_train, y_train)
    
    # 预测并计算测试集 RMSE
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv_rmse_scores.append(rmse)

# 4. 输出统计结果
cv_rmse_scores = np.array(cv_rmse_scores)
target_threshold = 0.15  # Erxn 阈值通常设为 0.15 eV

print("=== 从 54 个数据抽 18 个，并在 18 个内部拆分 (14 训练 vs 4 测试) 结果 ===")
print(f"模拟总次数: {len(cv_rmse_scores)}")
print(f"平均测试集 RMSE: {cv_rmse_scores.mean():.4f} eV")
print(f"RMSE 标准差: {cv_rmse_scores.std():.4f} eV")
print(f"最小 RMSE: {cv_rmse_scores.min():.4f} eV")
print(f"最大 RMSE: {cv_rmse_scores.max():.4f} eV")
print(f"RMSE < {target_threshold} eV 的达标次数: {np.sum(cv_rmse_scores < target_threshold)} / 100")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

# 1. 读取 54 个完整数据并彻底解绑原始索引
df = pd.read_excel('EG-3.xlsx').reset_index(drop=True)

# 固定前 18 个 AL 初始数据，切出剩余 36 个数据并重置索引
df_initial_18 = df.iloc[:18].copy()
df_remaining_36 = df.iloc[18:].copy().reset_index(drop=True)

# 2. 定义特征 X 和目标 y 的提取函数
def get_X_y(data_frame, target_col='Erxn'):
    # drop 掉所有非特征列/目标列（确保模型不发生特征泄漏）
    X = data_frame.drop(columns=['Erxn', 'Eads', 'Name'], errors='ignore')
    y = data_frame[target_col]
    return X, y

cv_rmse_scores = []

# 3. 蒙特卡洛 100 次随机抽样模拟
for i in range(100):
    # 第一步：从剩余 36 个数据中无放回随机抽取 18 个
    df_random_18, _ = train_test_split(
        df_remaining_36, 
        train_size=18, 
        random_state=i, 
        shuffle=True
    )
    
    # 第二步：拼接【固定前 18 个】+【随机抽取的 18 个】= 36 个数据集
    df_combined_36 = pd.concat([df_initial_18, df_random_18], axis=0).reset_index(drop=True)
    
    # 第三步：将这 36 个数据内部划分为训练集和测试集（此处以 8:2 划分为例，也可自定义 test_size）
    df_train, df_test = train_test_split(
        df_combined_36, 
        test_size=0.2,       # 示例：约 29 个作为训练集，7 个作为测试集
        random_state=i, 
        shuffle=True
    )
    
    # 提取特征与目标变量
    X_train, y_train = get_X_y(df_train, target_col='Erxn') # 评估吸附能改为 'Eads'
    X_test, y_test = get_X_y(df_test, target_col='Erxn')
    
    # 4. 初始化 GBR 模型并训练
    model = GradientBoostingRegressor(random_state=i)
    model.fit(X_train, y_train)
    
    # 5. 预测并计算测试集 RMSE
    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    cv_rmse_scores.append(rmse)

# 4. 统计分析与输出结果
cv_rmse_scores = np.array(cv_rmse_scores)
target_threshold = 0.15  # Erxn 达标阈值通常设为 0.15 eV

print("=== 18固定初始 + 18随机补充 -> 36个样本内部划分为训练/测试集 ===")
print(f"模拟总次数: {len(cv_rmse_scores)}")
print(f"平均测试集 RMSE: {cv_rmse_scores.mean():.4f} eV")
print(f"RMSE 标准差: {cv_rmse_scores.std():.4f} eV")
print(f"最小 RMSE: {cv_rmse_scores.min():.4f} eV")
print(f"最大 RMSE: {cv_rmse_scores.max():.4f} eV")
print(f"RMSE < {target_threshold} eV 的达标次数: {np.sum(cv_rmse_scores < target_threshold)} / 100")